# Processing and Export

In this example, we will process a series of measurements in order to save the output and metrics.

In [ ]:
import tempfile
from pathlib import Path

import piprkit

## Cleaning, splitting and baseline correction

Previous steps are repeated.

In [ ]:
ps = piprkit.load_real_series()
print(f"load_real_series(): {ps.get_samples()} samples, {ps.get_nan()} NaN")
ps.filters.rolling_mean(0.1)
print(f"rolling_mean(0.1): {ps.get_samples()} samples, {ps.get_nan()} NaN")
ps.filters.limit_rate_of_change()
print(f"limit_rate_of_change(): {ps.get_samples()} samples, {ps.get_nan()} NaN")
ps.drop_nan()
print(f"drop_nan(): {ps.get_samples()} samples, {ps.get_nan()} NaN")
ps.add_blinks([(153.5, 156.5)])
ps.filters.remove_blinks()
print(f"remove_blinks(): {ps.get_samples()} samples, {ps.get_nan()} NaN")
ps.trim_size()
print(f"trim_size(): {ps.get_samples()} samples, {ps.get_nan()} NaN")

trials = ps.split(pre_stimulus_duration=10.0, post_stimulus_duration=55.0)
for number, trial in enumerate(trials, start=1):
    trial.filters.apply_baseline_correction(duration=5.0)
    print(f"trial {number}: {trial.get_samples()} samples, {trial.get_nan()} NaN")

## Saving a trial

Using `piprkit.save_csv` trials can be saved to a csv file, with light stimulus and blinks in a JSON sidecar.

In [ ]:
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "trial_3.csv"
    piprkit.save_csv(trials[2], path)
    print(sorted(file.name for file in Path(folder).iterdir()))
    print(path.with_suffix(".json").read_text())

## Exporting the metrics

Using `piprkit.metric_rows` the results can be prepared to add to a list, that can subsequently be saved using `piprkit.save_metric_rows`.

In [ ]:
rows = []
for number, trial in enumerate(trials, start=1):
    results = [trial.metrics.pipr_6s(), trial.metrics.peak_constriction()]
    rows += piprkit.metric_rows(results, participant="example", trial=number)

with tempfile.TemporaryDirectory() as folder:
    folder = Path(folder)
    piprkit.save_metric_rows(rows, folder / "metrics.csv")
    print((folder / "metrics.csv").read_text())